In [1]:
import deepchem as dc
import numpy as np
import pandas as pd

print("deepchem:", dc.__version__)

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (/Users/reid/py/deepchemBenchmark/.venv/lib/python3.13/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-geome

deepchem: 2.8.0


## Load ChEMBL from MoleculeNet

DeepChem exposes `load_chembl` (older ChEMBL snapshot, multitask regression on bioactivity)
and `load_chembl25` (ChEMBL v25). We use a cheap featurizer (`CircularFingerprint`) here so
loading is fast; swap in a heavier featurizer once the schema is understood.

`featurizer="ECFP"` is a shorthand accepted by most molnet loaders and is equivalent to
`dc.feat.CircularFingerprint(size=1024)`.

In [2]:
tasks, datasets, transformers = dc.molnet.load_chembl(
    featurizer="ECFP",
    set_name="sparse",   # "sparse" (multitask, mostly-missing labels) or "5thresh"
    split="random",
    reload=True,         # cache to ~/.deepchem
)
train, valid, test = datasets
print("tasks:", len(tasks))
print("train / valid / test sizes:", len(train), len(valid), len(test))

'split' is deprecated.  Use 'splitter' instead.
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:45] DEPRECATION WARNING: please use MorganGenerator
[17:07:4

tasks: 691
train / valid / test sizes: 19096 2387 2388


## Dataset shape at a glance

- `X` — featurized inputs, shape `(n_samples, feature_dim)`
- `y` — labels, shape `(n_samples, n_tasks)`
- `w` — per-sample-per-task weights; `w == 0` marks *missing* labels (crucial for ChEMBL, which is very sparse)
- `ids` — one string per sample (SMILES for ChEMBL loaders)

In [3]:
print("X    :", train.X.shape, train.X.dtype)
print("y    :", train.y.shape, train.y.dtype)
print("w    :", train.w.shape, train.w.dtype)
print("ids  :", train.ids.shape, train.ids.dtype)
print("first id (SMILES?):", train.ids[0])

X    : (19096, 1024) float64
y    : (19096, 691) float64
w    : (19096, 691) float64
ids  : (19096,) object
first id (SMILES?): Cc1cccc(N2CCN(CCCCNC(=O)c3oc4ccccc4c3)CC2)c1C


In [4]:
# Task names — each column of y corresponds to one bioactivity target
print("first 10 tasks:", tasks[:10])
print("total tasks   :", len(tasks))

first 10 tasks: ['CHEMBL1075051', 'CHEMBL1075104', 'CHEMBL1075145', 'CHEMBL1075189', 'CHEMBL1075228', 'CHEMBL1075284', 'CHEMBL1075319', 'CHEMBL1163101', 'CHEMBL1163116', 'CHEMBL1163125']
total tasks   : 691


## Label sparsity

ChEMBL is a highly-sparse multitask problem: most (molecule, task) pairs have no measurement.
DeepChem encodes "no measurement" as `w == 0` (the `y` entry is a placeholder). Never trust
`y` without also checking `w`.

In [5]:
w = train.w
observed_mask = (w != 0)

n_cells = w.size
n_observed = int(observed_mask.sum())
print(f"observed labels: {n_observed:,} / {n_cells:,} ({100*n_observed/n_cells:.2f}%)")

# Per-task counts
per_task_obs = observed_mask.sum(axis=0)
print(f"labels per task -- min={per_task_obs.min()}, "
      f"median={int(np.median(per_task_obs))}, max={per_task_obs.max()}")

# Per-molecule counts
per_mol_obs = observed_mask.sum(axis=1)
print(f"tasks measured per molecule -- min={per_mol_obs.min()}, "
      f"median={int(np.median(per_mol_obs))}, max={per_mol_obs.max()}")

observed labels: 72,847 / 13,195,336 (0.55%)
labels per task -- min=0, median=31, max=1739
tasks measured per molecule -- min=3, median=3, max=97


In [6]:
# Distribution of observed y values (pooled across tasks).
# Note: `transformers` were fit on train, so these values are normalized.
observed_y = train.y[observed_mask]
print("observed y (normalized):",
      f"min={observed_y.min():.3f}",
      f"mean={observed_y.mean():.3f}",
      f"std={observed_y.std():.3f}",
      f"max={observed_y.max():.3f}")
print("transformers applied:", [type(t).__name__ for t in transformers])

observed y (normalized): min=1.632 mean=9.389 std=8.382 max=138.185
transformers applied: ['NormalizationTransformer']


## SMILES + IDs

For molnet loaders that start from SMILES (ChEMBL included), `dataset.ids` holds the SMILES
string per sample. There is no separate ChEMBL-ID column exposed by the loader — if you need
`CHEMBL####` IDs you have to cross-reference the raw ChEMBL release yourself.

In [7]:
sample = pd.DataFrame({
    "smiles": train.ids[:10],
    "n_observed_tasks": per_mol_obs[:10],
})
sample

,smiles,n_observed_tasks
0,Cc1cccc(N2CCN(CCCCNC(=O)c3oc4ccccc4c3)CC2)c1C,5
1,OC(=O)c1cccc(O)c1C(=O)c2c(O)cc(cc2O)C(=O)OC3CN...,4
2,CC(=O)OC[C@H]1O[C@@H](SCc2cn(nn2)c3ccc(cc3)S(=...,4
3,CCCC(=O)NO,5
4,COc1ccc(CC(=O)N2CCC3(CC2)CN(C3)[C@@H]4CCc5cc(c...,3
5,CNc1ncc2cc(ccc2n1)c3cc(Nc4nc5cc(ccc5[nH]4)C(F)...,3
6,NC(=O)c1ccc(Oc2ccc(CN3CCC[C@H]3c4cccnc4)cc2)c(...,3
7,O=C(NCCCN1CCC2(CC1)CCc3ccccc23)[C@H]4CCCN4Cc5c...,3
8,CC1(C)Oc2cc(cc(O)c2[C@@H]3C[C@H](O)CC[C@@H]13)...,3
9,Cc1cccc(c1)N2CCN(CC2)C(=O)Nc3ccc(OS(=O)(=O)N)cc3,4


In [8]:
# Sanity-check that ids are valid SMILES by round-tripping through RDKit.
from rdkit import Chem
bad = [s for s in train.ids[:200] if Chem.MolFromSmiles(s) is None]
print(f"invalid SMILES in first 200 ids: {len(bad)}")

invalid SMILES in first 200 ids: 0


## Peek at a single task

Pick the task with the most observations and look at its label distribution + a few example
molecules.

In [9]:
top_task_idx = int(per_task_obs.argmax())
top_task = tasks[top_task_idx]
mask = observed_mask[:, top_task_idx]
print(f"task: {top_task}  (idx={top_task_idx}, n_observed={mask.sum()})")

y_task = train.y[mask, top_task_idx]
print(f"y stats: min={y_task.min():.3f}  mean={y_task.mean():.3f}  max={y_task.max():.3f}")

pd.DataFrame({
    "smiles": train.ids[mask][:10],
    "y_normalized": y_task[:10],
})

task: CHEMBL261  (idx=237, n_observed=1739)
y stats: min=1.854  mean=3.103  max=5.642


,smiles,y_normalized
0,CC(=O)OC[C@H]1O[C@@H](SCc2cn(nn2)c3ccc(cc3)S(=...,3.499020
1,Cc1cccc(c1)N2CCN(CC2)C(=O)Nc3ccc(OS(=O)(=O)N)cc3,2.974076
2,CC(C)(C)OC(=O)NCCCCOc1ccc(cc1)S(=O)(=O)N,3.688433
3,COc1ccc(\C=C(/C#N)\C(=O)Nc2ccc(cc2)S(=O)(=O)N)cc1,3.693845
4,COc1cc2C=CC(=O)Oc2c(O)c1OC,2.395014
5,NC(=O)CN1C=CC(=O)C(=C1)S(=O)(=O)N,2.519485
6,NS(=O)(=O)c1ccc(OCCCCO[N+](=O)[O-])cc1,3.688433
7,NS(=O)(=O)c1ccc(N2C(=O)c3c(Cl)c(Cl)c(Cl)c(Cl)c...,3.168900
8,Cc1cc(C)nc(Sc2c(F)c(F)c(c(F)c2F)S(=O)(=O)N)n1,4.343260
9,CCCN(C)c1nc(NCCc2ccc(cc2)S(=O)(=O)N)nc(n1)N(C)CCC,3.315019


In [10]:
# Optional: un-transform y back to the original label scale for that one task.
y_task_col = np.zeros_like(train.y)
y_task_col[mask, top_task_idx] = y_task
w_task_col = np.zeros_like(train.w)
w_task_col[mask, top_task_idx] = 1.0

restored = train.y.copy()
for t in reversed(transformers):
    restored = t.untransform(restored)
print("untransformed y range for top task:",
      restored[mask, top_task_idx].min(),
      restored[mask, top_task_idx].max())

untransformed y range for top task: 4.0 11.0


In [11]:
train.get_statistics()

(array([0.01838081, 0.27906368, 0.06200251, ..., 0.02576456, 0.0164956 ,
        0.00858819], shape=(1024,)),
 array([0.13432408, 0.4485389 , 0.24116012, ..., 0.15843215, 0.12737149,
        0.09227366], shape=(1024,)),
 array([-7.65082510e-19, -3.83705925e-19,  0.00000000e+00,  0.00000000e+00,
        -1.07959855e-16, -1.61276132e-16,  6.11187211e-17, -8.41878752e-17,
         0.00000000e+00,  4.49730026e-17, -4.21229485e-18,  0.00000000e+00,
        -2.13186546e-17, -5.97158228e-20, -1.13886227e-17, -3.54534110e-17,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  6.20753389e-17,
        -2.24829014e-17,  1.54730685e-16,  0.00000000e+00, -2.24780945e-16,
         4.17582161e-16,  1.72792921e-16,  3.77937631e-17, -5.38665309e-18,
         3.67679215e-16, -2.66874671e-16, -3.02136652e-18,  1.86981253e-16,
        -3.45583513e-16,  0.00000000e+00, -1.75712750e-16,  0.00000000e+00,
        -9.74858689e-17,  8.02936412e-17, -8.28872561e-17,  4.50099332e-16,
        -2.69889261e

## Sample-0 JSON record

Bundle the first training sample's SMILES, its 1024-bit ECFP vector, and every observed
`y` value into a single JSON object. For each task with `w != 0` we add a top-level entry
keyed by the task name and valued by the corresponding `y`.

In [17]:
tasks[task_idx]

'CHEMBL214'

In [29]:
tasks

['CHEMBL1075051',
 'CHEMBL1075104',
 'CHEMBL1075145',
 'CHEMBL1075189',
 'CHEMBL1075228',
 'CHEMBL1075284',
 'CHEMBL1075319',
 'CHEMBL1163101',
 'CHEMBL1163116',
 'CHEMBL1163125',
 'CHEMBL1255149',
 'CHEMBL1255150',
 'CHEMBL1293255',
 'CHEMBL1293289',
 'CHEMBL1293292',
 'CHEMBL1741186',
 'CHEMBL1741195',
 'CHEMBL1744525',
 'CHEMBL1764940',
 'CHEMBL1781',
 'CHEMBL1781862',
 'CHEMBL1782',
 'CHEMBL1784',
 'CHEMBL1790',
 'CHEMBL1792',
 'CHEMBL1795101',
 'CHEMBL1795126',
 'CHEMBL1800',
 'CHEMBL1801',
 'CHEMBL1804',
 'CHEMBL1806',
 'CHEMBL1811',
 'CHEMBL1821',
 'CHEMBL1822',
 'CHEMBL1824',
 'CHEMBL1825',
 'CHEMBL1827',
 'CHEMBL1829',
 'CHEMBL1833',
 'CHEMBL1836',
 'CHEMBL1844',
 'CHEMBL1849',
 'CHEMBL1850',
 'CHEMBL1853',
 'CHEMBL1855',
 'CHEMBL1856',
 'CHEMBL1860',
 'CHEMBL1862',
 'CHEMBL1865',
 'CHEMBL1867',
 'CHEMBL1868',
 'CHEMBL1871',
 'CHEMBL1873',
 'CHEMBL1875',
 'CHEMBL1878',
 'CHEMBL1881',
 'CHEMBL1889',
 'CHEMBL1892',
 'CHEMBL1898',
 'CHEMBL1899',
 'CHEMBL1900',
 'CHEMBL1901',
 'CH

In [28]:
train.tasks

array(['CHEMBL1075051', 'CHEMBL1075104', 'CHEMBL1075145', 'CHEMBL1075189',
       'CHEMBL1075228', 'CHEMBL1075284', 'CHEMBL1075319', 'CHEMBL1163101',
       'CHEMBL1163116', 'CHEMBL1163125', 'CHEMBL1255149', 'CHEMBL1255150',
       'CHEMBL1293255', 'CHEMBL1293289', 'CHEMBL1293292', 'CHEMBL1741186',
       'CHEMBL1741195', 'CHEMBL1744525', 'CHEMBL1764940', 'CHEMBL1781',
       'CHEMBL1781862', 'CHEMBL1782', 'CHEMBL1784', 'CHEMBL1790',
       'CHEMBL1792', 'CHEMBL1795101', 'CHEMBL1795126', 'CHEMBL1800',
       'CHEMBL1801', 'CHEMBL1804', 'CHEMBL1806', 'CHEMBL1811',
       'CHEMBL1821', 'CHEMBL1822', 'CHEMBL1824', 'CHEMBL1825',
       'CHEMBL1827', 'CHEMBL1829', 'CHEMBL1833', 'CHEMBL1836',
       'CHEMBL1844', 'CHEMBL1849', 'CHEMBL1850', 'CHEMBL1853',
       'CHEMBL1855', 'CHEMBL1856', 'CHEMBL1860', 'CHEMBL1862',
       'CHEMBL1865', 'CHEMBL1867', 'CHEMBL1868', 'CHEMBL1871',
       'CHEMBL1873', 'CHEMBL1875', 'CHEMBL1878', 'CHEMBL1881',
       'CHEMBL1889', 'CHEMBL1892', 'CHEMBL1898', 'CH

In [56]:
import json

# Pick the first sample that has at least one observed task, so y is a real measurement.
has_observation = (train.w != 0).any(axis=1)
candidates = np.flatnonzero(has_observation)
if candidates.size == 0:
    raise RuntimeError("No sample in train has any observed task; check the loader output.")
i = int(candidates[0])

x_vec = train.X[i]
w_row = train.w[i]
y_row = train.y[i]

record = {
    "smiles": str(train.ids[i]),
    "X":      x_vec.astype(int).tolist(),   # 1024-length ECFP bit vector
}

# One entry per observed task: key = task name, value = y for that task.
observed_task_indices = np.flatnonzero(w_row != 0)
for t_idx in observed_task_indices:
    record[tasks[int(t_idx)]] = float(y_row[t_idx])

record_json = json.dumps(record)
print(f"JSON size: {len(record_json):,} chars")

JSON size: 3,297 chars


In [59]:
from pprint import pprint 
pprint(record)

{'CHEMBL214': 5.182933408873236,
 'CHEMBL217': 3.91905576465042,
 'CHEMBL224': 5.317881355992391,
 'CHEMBL225': 6.245883257904363,
 'CHEMBL234': 6.222473937546522,
 'X': [0,
       0,
       0,
       0,
       0,
       0,
       1,
       0,
       1,
       0,
       0,
       0,
       0,
       0,
       0,
       1,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       1,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       1,
       1,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       0,
       1,
       0,
       0,
       0,
       0,
       0,
       1,
       0,
       0,
      

In [ ]:
# Optionally persist to disk.
with open("sample0.json", "w") as f:
    json.dump(record, f)

## Translate task IDs → biomolecular targets

DeepChem's ChEMBL task names are ChEMBL **target** identifiers (e.g. `CHEMBL204`).
We use the official `chembl_webresource_client` (batches + local cache) to translate
each task into:

- `pref_name` — human-readable name (e.g. *Prothrombin*, *Dopamine D2 receptor*)
- `target_type` — coarse bucket (`SINGLE PROTEIN`, `PROTEIN COMPLEX`, `CELL-LINE`, `ORGANISM`, …)
- `organism`
- `uniprot` — accession(s) for cross-reference
- Protein-classification hierarchy `l1 … l5` — the most interpretable grouping

Install with `uv add chembl_webresource_client` if it isn't already.

In [38]:
from chembl_webresource_client.new_client import new_client

def translate_tasks(task_ids: list[str]) -> pd.DataFrame:
    """Batch-translate ChEMBL target IDs into names + protein-class hierarchy."""
    task_ids = list(task_ids)

    # 1. Batch fetch target metadata + their components.
    targets = list(new_client.target.filter(target_chembl_id__in=task_ids))
    target_by_id = {t["target_chembl_id"]: t for t in targets}

    all_component_ids = {
        c["component_id"]
        for t in targets
        for c in t.get("target_components", [])
    }

    # 2. Batch fetch component detail (this is where protein_classifications live).
    components = list(
        new_client.target_component.filter(component_id__in=list(all_component_ids))
    )
    comp_by_id = {c["component_id"]: c for c in components}

    all_class_ids = {
        pc["protein_classification_id"]
        for c in components
        for pc in c.get("protein_classifications", [])
    }

    # 3. Batch fetch the classification nodes; protein_class_desc has the full L1..Ln lineage.
    class_nodes = list(
        new_client.protein_classification.filter(protein_class_id__in=list(all_class_ids))
    )
    class_by_id = {c["protein_class_id"]: c for c in class_nodes}

    rows = []
    for tid in task_ids:
        t = target_by_id.get(tid)
        row = {
            "task":     tid,
            "name":     None,
            "type":     None,
            "organism": None,
            "uniprot":  None,
            "l1": None, "l2": None, "l3": None, "l4": None, "l5": None,
            "class_path": None,
        }
        if t is None:
            rows.append(row)
            continue

        row["name"]     = t.get("pref_name")
        row["type"]     = t.get("target_type")
        row["organism"] = t.get("organism")

        # First protein component gets to define uniprot + protein class.
        for comp_ref in t.get("target_components", []):
            comp = comp_by_id.get(comp_ref["component_id"], {})
            if comp.get("component_type") != "PROTEIN":
                continue
            row["uniprot"] = comp.get("accession")
            pcs = comp.get("protein_classifications", [])
            if pcs:
                node = class_by_id.get(pcs[0]["protein_classification_id"], {})
                # protein_class_desc is space-separated L1..Ln in lowercase.
                levels = (node.get("protein_class_desc") or "").split()
                for i, lvl in enumerate(levels[:5], start=1):
                    row[f"l{i}"] = lvl
                row["class_path"] = " > ".join(levels) or None
            break
        rows.append(row)

    return pd.DataFrame(rows)

In [49]:
# Translate every observed task for sample i.
observed_task_ids = [tasks[int(t)] for t in observed_task_indices]
df_targets = translate_tasks(observed_task_ids)

## Interpretable groupings

- **`type`** — coarsest: `SINGLE PROTEIN`, `PROTEIN COMPLEX`, `CELL-LINE`, `ORGANISM`, …
- **`l1`** — biology top-level: `enzyme`, `membrane receptor`, `transcription factor`, `ion channel`, `transporter`, `epigenetic regulator`, …
- **`l2`** — the useful working level: `enzyme > kinase / protease / oxidoreductase`, `membrane receptor > family a g protein-coupled receptor`, …

In [50]:
df_targets.groupby(["type", "l1"], dropna=False).agg(
    n_tasks=("task", "count"),
    examples=("name", lambda s: list(s)[:3]),
)

,,n_tasks,examples
type,l1,,
SINGLE PROTEIN,membrane,5,"[5-hydroxytryptamine receptor 1A, D(2) dopamin..."


In [47]:
# Larger slice — useful for understanding the whole benchmark, not just one sample.
# Bump the slice size once you're comfortable with the request time.
sample_ids = list(tasks[:200])
df_all = translate_tasks(sample_ids)

df_all.groupby(["l1", "l2"], dropna=False).size().sort_values(ascending=False).head(30)

l1             l2               
membrane       receptor             65
enzyme         kinase               25
               reductase            20
               protease             19
transcription  factor               14
enzyme         NaN                   9
ion            channel               7
enzyme         isomerase             6
epigenetic     regulator             5
transporter    electrochemical       5
enzyme         hydrolase             5
               transferase           4
               phosphodiesterase     4
auxiliary      transport             2
enzyme         lyase                 2
               cytochrome            2
cytosolic      other                 1
membrane       other                 1
secreted       NaN                   1
enzyme         ligase                1
transporter    ntpase                1
unclassified   NaN                   1
dtype: int64

In [52]:
df_all.groupby(["l1", "l2"], dropna=False).size().sort_values(ascending=False)

l1             l2               
membrane       receptor             65
enzyme         kinase               25
               reductase            20
               protease             19
transcription  factor               14
enzyme         NaN                   9
ion            channel               7
enzyme         isomerase             6
epigenetic     regulator             5
transporter    electrochemical       5
enzyme         hydrolase             5
               transferase           4
               phosphodiesterase     4
auxiliary      transport             2
enzyme         lyase                 2
               cytochrome            2
cytosolic      other                 1
membrane       other                 1
secreted       NaN                   1
enzyme         ligase                1
transporter    ntpase                1
unclassified   NaN                   1
dtype: int64